# Step 7 (Stretch): RAG Layer for Plain-English Churn Explanations

**What we're doing:** The Telco dataset has no text data, so we generate
synthetic 'support ticket' text per customer based on their actual
features (e.g. a customer with no TechSupport and high MonthlyCharges
gets a synthetic complaint about cost/support). We then build a small RAG
(Retrieval-Augmented Generation) pipeline: embed each ticket, retrieve the
most relevant ones for a given customer, and feed those + the model's
churn prediction to an LLM, which writes a plain-English risk explanation.

**Why this is the differentiator:** plenty of repos predict churn. Almost
none combine that prediction with a retrieval pipeline and an LLM that
explains WHY in language a non-technical stakeholder (a retention rep, a
manager) can actually act on. This is also a legitimate, realistic
pattern — RAG over support tickets is a genuine enterprise use case.

**Honesty note for your README:** the support tickets are SYNTHETIC,
generated from the structured features as a stand-in for real
unstructured text a company might have (call transcripts, support
tickets, NPS comments). Be upfront about this — it's still a fully real,
working RAG pipeline; only the input text is simulated.

**Which LLM to use, free:** this notebook uses the Anthropic API
(Claude) as the generation step, since it's likely the API you already
have a key for. Swap in any other API (OpenAI, a free Hugging Face
inference endpoint, or a small local model) if you prefer — the RAG
pipeline logic before that step doesn't change.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/churn-platform'

!pip install -q sentence-transformers anthropic

import pandas as pd
import numpy as np
import joblib

df = pd.read_csv(f'{PROJECT_ROOT}/data/processed/churn_clean.csv')
model = joblib.load(f'{PROJECT_ROOT}/models/xgb_model.joblib')
feature_columns = joblib.load(f'{PROJECT_ROOT}/models/feature_columns.joblib')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.7/960.7 kB 42.6 MB/s eta 0:00:00


/usr/lib/python3.12/pickle.py:1760: UserWarning: [05:50:00] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


## 7.1 Generate synthetic support tickets

**Why rule-based generation, not just random text:** the tickets need to
be GROUNDED in each customer's actual feature values, so the later
retrieval + explanation step is internally consistent (a high-MonthlyCharges,
no-TechSupport customer should get a ticket mentioning cost/support
frustration — not a randomly unrelated complaint). This keeps the demo
honest about what it's simulating.

In [2]:
def generate_ticket(row):
    """Builds a plausible synthetic support-ticket sentence from a
    customer's real feature values."""
    phrases = []

    if row['Contract'] == 'Month-to-month':
        phrases.append("Customer asked about switching to a cheaper plan or canceling since they're not on a long-term contract.")
    if row['TechSupport'] == 'No' and row['InternetService'] != 'No':
        phrases.append("Customer reported frustration with unresolved internet issues and no tech support add-on.")
    if row['MonthlyCharges'] > 80:
        phrases.append("Customer complained that their monthly bill feels too high compared to competitors.")
    if row['tenure'] < 6:
        phrases.append("Customer is new and asked several onboarding questions, seems still evaluating the service.")
    if row['PaymentMethod'] == 'Electronic check':
        phrases.append("Customer asked about changing their payment method, mentioned billing confusion.")
    if row['OnlineSecurity'] == 'No':
        phrases.append("Customer expressed concern about lack of online security features.")

    if not phrases:
        phrases.append("Customer contacted support with a general account question; no specific complaints noted.")

    return " ".join(phrases)

df['synthetic_ticket'] = df.apply(generate_ticket, axis=1)
df[['customerID', 'synthetic_ticket']].head(3)

,customerID,synthetic_ticket
0,7590-VHVEG,Customer asked about switching to a cheaper pl...
1,5575-GNVDE,Customer reported frustration with unresolved ...
2,3668-QPYBK,Customer asked about switching to a cheaper pl...


## 7.2 Embed the tickets

**Why embeddings + retrieval at all, for a single customer's own ticket:**
in this simplified version, retrieval finds SIMILAR customers' tickets —
useful context like "here's how other high-risk customers with similar
complaints were described." This is the core RAG pattern (retrieve
relevant context, then generate) even though our corpus is synthetic.
In a real system, this corpus would be actual historical tickets, and
retrieval would surface genuinely informative precedent.

In [3]:
from sentence_transformers import SentenceTransformer

# A small, fast, free, locally-run embedding model — no API key needed
# for this step, which keeps the RAG retrieval cost-free.
embedder = SentenceTransformer('all-MiniLM-L6-v2')

ticket_embeddings = embedder.encode(df['synthetic_ticket'].tolist(), show_progress_bar=True)
print('Embedding matrix shape:', ticket_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/221 [00:00<?, ?it/s]

Embedding matrix shape: (7043, 384)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_similar_tickets(customer_idx, top_k=3):
    """Finds the top_k most similar tickets to the given customer's ticket,
    excluding the customer's own ticket."""
    query_vec = ticket_embeddings[customer_idx].reshape(1, -1)
    sims = cosine_similarity(query_vec, ticket_embeddings)[0]
    sims[customer_idx] = -1  # exclude self
    top_indices = sims.argsort()[::-1][:top_k]
    return df.iloc[top_indices][['customerID', 'synthetic_ticket']], sims[top_indices]

## 7.3 Get the model's churn prediction for a sample customer
We pick one customer to walk through end to end: prediction → retrieval →
LLM explanation.

In [5]:
sample_idx = df[df['Churn'] == 1].index[0]   # pick an actual churned customer
sample_customer = df.loc[sample_idx]

X_sample = pd.get_dummies(
    sample_customer.drop(['customerID', 'Churn', 'synthetic_ticket']).to_frame().T
).reindex(columns=feature_columns, fill_value=0)

churn_prob = model.predict_proba(X_sample)[:, 1][0]
print(f"Customer {sample_customer['customerID']} — predicted churn probability: {churn_prob:.3f}")
print(f"Their ticket: {sample_customer['synthetic_ticket']}")

similar_tickets, sims = retrieve_similar_tickets(sample_idx, top_k=3)
print('\nMost similar tickets from other customers:')
for (_, row), sim in zip(similar_tickets.iterrows(), sims):
    print(f"  [{sim:.2f}] {row['synthetic_ticket']}")

Customer 3668-QPYBK — predicted churn probability: 0.278
Their ticket: Customer asked about switching to a cheaper plan or canceling since they're not on a long-term contract. Customer reported frustration with unresolved internet issues and no tech support add-on. Customer is new and asked several onboarding questions, seems still evaluating the service.

Most similar tickets from other customers:
  [1.00] Customer asked about switching to a cheaper plan or canceling since they're not on a long-term contract. Customer reported frustration with unresolved internet issues and no tech support add-on. Customer is new and asked several onboarding questions, seems still evaluating the service.
  [1.00] Customer asked about switching to a cheaper plan or canceling since they're not on a long-term contract. Customer reported frustration with unresolved internet issues and no tech support add-on. Customer is new and asked several onboarding questions, seems still evaluating the service.
  [1.0

## 7.4 Generate a plain-English explanation with an LLM

**Why feed the LLM structured features + retrieved tickets, not just ask
it to predict churn itself:** the LLM is NOT the predictor here — the
XGBoost model already made the prediction. The LLM's job is narrowly
scoped to EXPLAIN a prediction that's already been made, grounded in real
feature values and retrieved context. This division of labor (statistical
model predicts, LLM explains) is the right way to combine the two — it
avoids relying on an LLM for the thing it's worst at (precise numeric
prediction) and uses it for the thing it's best at (fluent explanation).

In [8]:
!pip install -q transformers

from transformers import pipeline

# Using text-generation task which works with the current transformers version
generator = pipeline("text-generation", model="gpt2")

def explain_churn_risk(customer_row, churn_prob, similar_tickets_df):
    feature_summary = (
        f"Contract: {customer_row['Contract']}, "
        f"Tenure: {customer_row['tenure']} months, "
        f"Monthly charges: ${customer_row['MonthlyCharges']}, "
        f"Internet service: {customer_row['InternetService']}, "
        f"Tech support: {customer_row['TechSupport']}, "
        f"Payment method: {customer_row['PaymentMethod']}"
    )

    similar_text = " ".join(
        t for t in similar_tickets_df['synthetic_ticket']
    )

    prompt = (
        f"Customer retention note: This customer is at risk of leaving. "
        f"Profile: {feature_summary}. "
        f"Their complaint: {customer_row['synthetic_ticket']}. "
        f"Recommendation:"
    )

    result = generator(
        prompt,
        max_new_tokens=100,
        num_return_sequences=1,
        pad_token_id=50256,   # stops a warning about padding
        truncation=True
    )

    # gpt2 returns the full text including the prompt — we only want
    # the part AFTER "Recommendation:" which is the generated bit
    full_text = result[0]['generated_text']
    generated_part = full_text.split("Recommendation:")[-1].strip()
    return generated_part

explanation = explain_churn_risk(sample_customer, churn_prob, similar_tickets)
print(explanation)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Do not rely on Internet speeds for your business. Customer reports no signs of frustration with internet speeds. Customer is happy to pay for internet and is happy to use their services. Customer is working hard and is happy to work with their ISP. Customer is happy to work with their ISP. Customer is working hard and is happy to work with their ISP. Customer is happy to work with their ISP. Customer is working hard and is happy to work with their ISP. Customer is working hard and is happy to


## 7.5 Run this for a batch of high-risk customers
Turns the one-off demo above into something resembling a real retention
report — a list of top at-risk customers, each with a plain-English reason.

In [9]:
X_all = pd.get_dummies(df.drop(columns=['customerID', 'Churn', 'synthetic_ticket']))
X_all = X_all.reindex(columns=feature_columns, fill_value=0)
df['churn_probability'] = model.predict_proba(X_all)[:, 1]

top_risk = df.sort_values('churn_probability', ascending=False).head(5)

report_rows = []
for idx, row in top_risk.iterrows():
    similar, _ = retrieve_similar_tickets(idx, top_k=2)
    explanation = explain_churn_risk(row, row['churn_probability'], similar)
    report_rows.append({
        'customerID': row['customerID'],
        'churn_probability': round(row['churn_probability'], 3),
        'explanation': explanation,
    })

report_df = pd.DataFrame(report_rows)
report_df.to_csv(f'{PROJECT_ROOT}/reports/top_risk_explanations.csv', index=False)
report_df

[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

,customerID,churn_probability,explanation
0,7216-EWTRS,0.955,Customer is very happy with the service. Custo...
1,5419-JPRRN,0.934,"If you're using a prepaid plan, make sure you'..."
2,9497-QCMMS,0.932,"Review of the service is essential, customers ..."
3,9300-AGZNL,0.932,This customer is at risk of leaving. Profile: ...
4,1069-XAIEM,0.929,"Contact Customer Support, and inform them of y..."
